In [ ]:
!pip install gtts librosa -q

In [ ]:
import random

# Stop words to make speech natural
stop_words = [
    "the", "a", "an", "this", "that", "is", "are", "was",
    "in", "on", "at", "with", "and", "of", "for", "to",
    "showing", "featuring", "captured", "seen", "during"
]

# News-style sentence templates
templates = [
    "this is {adj1} footage of {cls} {action}",
    "the video is showing {adj1} {cls} {action} in {place}",
    "captured footage featuring {cls} {action} with {adj2} conditions",
    "this {adj1} video is of {cls} {action} during {time}",
    "footage showing {adj1} and {adj2} {cls} {action}",
]

# Class specific words
class_speech_words = {
    'Basketball' : {"adj1": ["professional","competitive","indoor","live"],
                    "adj2": ["intense","exciting","fast","dynamic"],
                    "action": ["players dribbling and shooting","match in progress","game underway"],
                    "place": ["indoor court","basketball arena","sports hall"],
                    "time":  ["the match","halftime","championship game"]},

    'Biking'     : {"adj1": ["outdoor","fast","competitive","scenic"],
                    "adj2": ["challenging","rough","open","clear"],
                    "action": ["rider cycling on road","person riding bicycle","cyclist in motion"],
                    "place": ["open road","cycling trail","outdoor track"],
                    "time":  ["the race","morning ride","training session"]},

    'Bowling'    : {"adj1": ["indoor","competitive","professional","casual"],
                    "adj2": ["precise","focused","clear","controlled"],
                    "action": ["player throwing ball down lane","bowler aiming at pins","strike in progress"],
                    "place": ["bowling alley","indoor lane","sports center"],
                    "time":  ["the game","tournament","practice session"]},

    'CliffDiving': {"adj1": ["extreme","breathtaking","outdoor","dangerous"],
                    "adj2": ["high","rocky","steep","dramatic"],
                    "action": ["athlete jumping from cliff","diver leaping into water","freefall in progress"],
                    "place": ["rocky cliff","ocean shore","natural diving spot"],
                    "time":  ["the jump","competition","training"]},

    'GolfSwing'  : {"adj1": ["professional","outdoor","precise","calm"],
                    "adj2": ["focused","clean","open","green"],
                    "action": ["golfer swinging club","player hitting ball","golf shot in progress"],
                    "place": ["golf course","open fairway","green"],
                    "time":  ["the round","tournament","practice"]},

    'HorseRiding': {"adj1": ["outdoor","graceful","competitive","rural"],
                    "adj2": ["open","natural","calm","scenic"],
                    "action": ["rider on horseback","person galloping on horse","equestrian in motion"],
                    "place": ["open field","equestrian track","countryside"],
                    "time":  ["the race","training","competition"]},

    'Skiing'     : {"adj1": ["winter","fast","outdoor","snowy"],
                    "adj2": ["steep","cold","icy","downhill"],
                    "action": ["skier going down slope","person skiing at speed","downhill skiing in progress"],
                    "place": ["snowy mountain","ski slope","winter resort"],
                    "time":  ["the run","competition","training session"]},

    'Surfing'    : {"adj1": ["outdoor","exciting","coastal","extreme"],
                    "adj2": ["large","powerful","ocean","clear"],
                    "action": ["surfer riding wave","person on surfboard","wave surfing in progress"],
                    "place": ["ocean shore","coastal waters","beach"],
                    "time":  ["the surf","competition","morning session"]},

    'TennisSwing': {"adj1": ["competitive","outdoor","professional","fast"],
                    "adj2": ["precise","powerful","focused","clean"],
                    "action": ["player swinging racket","tennis shot in progress","serve and volley"],
                    "place": ["tennis court","outdoor court","sports arena"],
                    "time":  ["the match","tournament","practice session"]},

    'SkateBoarding':{"adj1": ["outdoor","extreme","urban","fast"],
                    "adj2": ["technical","smooth","urban","open"],
                    "action": ["skater performing tricks","person riding skateboard","skateboard in motion"],
                    "place": ["skate park","urban area","outdoor ramp"],
                    "time":  ["the session","competition","practice"]},
}

def generate_speech_text(cls, training=True):
    """Generate natural news-style speech with stop words"""
    words = class_speech_words[cls]

    # Pick random template
    template = random.choice(templates)

    # Fill template
    speech = template.format(
        cls    = cls.lower(),
        adj1   = random.choice(words["adj1"]),
        adj2   = random.choice(words["adj2"]),
        action = random.choice(words["action"]),
        place  = random.choice(words["place"]),
        time   = random.choice(words["time"])
    )

    # Add random stop words at random positions
    words_list = speech.split()
    num_stops  = random.randint(2, 4)
    for _ in range(num_stops):
        pos  = random.randint(0, len(words_list))
        stop = random.choice(stop_words)
        words_list.insert(pos, stop)

    final_speech = " ".join(words_list)
    return final_speech

# Test
print("Sample speech texts:")
for cls in selected_classes[:3]:
    s1 = generate_speech_text(cls)
    s2 = generate_speech_text(cls)
    print(f"\n{cls}:")
    print(f"  '{s1}'")
    print(f"  '{s2}'")

In [ ]:
from gtts import gTTS
import librosa
import numpy as np
import torch
import io
import os
import tempfile

def text_to_mfcc(text, n_mfcc=40, max_len=128):
    """
    Convert text → speech (gTTS) → MFCC features
    Returns tensor of shape (n_mfcc * 3,) = 120-dim
    """
    try:
        # Step 1 — Text to Speech
        tts = gTTS(text=text, lang='en', slow=False)

        # Save to temp file
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
            tmp_path = f.name
        tts.save(tmp_path)

        # Step 2 — Load audio
        audio, sr = librosa.load(tmp_path, sr=22050)
        os.unlink(tmp_path)

        # Step 3 — Extract MFCC
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

        # Pad or truncate to fixed length
        if mfcc.shape[1] < max_len:
            mfcc = np.pad(mfcc, ((0,0),(0, max_len - mfcc.shape[1])))
        else:
            mfcc = mfcc[:, :max_len]

        # Compute mean, std, max across time → (n_mfcc * 3,) = 120-dim
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std  = np.std(mfcc,  axis=1)
        mfcc_max  = np.max(mfcc,  axis=1)

        features = np.concatenate([mfcc_mean, mfcc_std, mfcc_max])
        return torch.FloatTensor(features)

    except Exception as e:
        print(f"Audio error: {e}")
        # Return zeros on failure
        return torch.zeros(n_mfcc * 3)

# Test
print("Testing audio feature extraction...")
test_text = generate_speech_text('Biking')
print(f"Speech: '{test_text}'")
features  = text_to_mfcc(test_text)
print(f"Audio feature shape: {features.shape}")
print(f"Feature sample: {features[:5]}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class TrimodalFusionModel(nn.Module):
    def __init__(self, num_classes=10, hidden_size=512, num_layers=2):
        super(TrimodalFusionModel, self).__init__()

        # --- Visual Branch (CNN + LSTM) ---
        resnet = models.resnet50(pretrained=True)
        for name, param in resnet.named_parameters():
            param.requires_grad = 'layer4' in name
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])

        self.lstm = nn.LSTM(
            input_size  = 2048,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = 0.3
        )
        self.visual_projector = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # --- Text Branch (DistilBERT) ---
        self.text_projector = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # --- Audio Branch (MFCC) ---
        # Input: 120-dim MFCC features
        self.audio_encoder = nn.Sequential(
            nn.Linear(120, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # --- Fusion Layer ---
        # Visual(256) + Text(256) + Audio(256) = 768
        self.fusion = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, video, text_feat, audio_feat):
        # --- Visual path ---
        batch_size, frames, C, H, W = video.shape
        video = video.view(batch_size * frames, C, H, W)

        with torch.no_grad():
            cnn_out = self.cnn(video)

        cnn_out     = cnn_out.view(batch_size, frames, -1)
        lstm_out, _ = self.lstm(cnn_out)
        visual_feat = self.visual_projector(lstm_out[:, -1, :])  # (batch, 256)

        # --- Text path ---
        if text_feat.dim() == 3:
            text_feat = text_feat.squeeze(1)
        text_feat = self.text_projector(text_feat)               # (batch, 256)

        # --- Audio path ---
        audio_feat = self.audio_encoder(audio_feat)              # (batch, 256)

        # --- Fusion ---
        fused = torch.cat([visual_feat, text_feat, audio_feat], dim=1)  # (batch, 768)
        out   = self.fusion(fused)
        return out

# Initialize
device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trimodal_model = TrimodalFusionModel(num_classes=10).to(device)
print("Trimodal Fusion Model ready!")
print(f"Trainable params: {sum(p.numel() for p in trimodal_model.parameters() if p.requires_grad):,}")

In [ ]:
import os
import random
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class TrimodalDataset(Dataset):
    def __init__(self, samples, selected_classes,
                 transform=None, training=False):
        self.samples          = samples
        self.selected_classes = selected_classes
        self.transform        = transform
        self.training         = training

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        cls_name          = self.selected_classes[label]

        # --- Video frames ---
        frames = sorted(os.listdir(video_path))
        frame_tensors = []
        for frame_file in frames:
            frame_path = os.path.join(video_path, frame_file)
            image      = Image.open(frame_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            frame_tensors.append(image)
        video_tensor = torch.stack(frame_tensors)

        # --- Text features ---
        if self.training:
            num_words = random.randint(4, 6)
            text_desc = generate_random_description(cls_name, num_words)
        else:
            random.seed(idx)
            text_desc = generate_random_description(cls_name, 5)
            random.seed()
        text_feat = get_text_features(text_desc).squeeze(0)

        # --- Audio features ---
        if self.training:
            speech_text = generate_speech_text(cls_name, training=True)
        else:
            random.seed(idx + 1000)
            speech_text = generate_speech_text(cls_name, training=False)
            random.seed()
        audio_feat = text_to_mfcc(speech_text)

        return video_tensor, text_feat, audio_feat, label

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Build samples
frames_path  = '/content/ucf_frames'
class_to_idx = {cls: idx for idx, cls in enumerate(selected_classes)}

all_samples = []
for cls in selected_classes:
    cls_path = os.path.join(frames_path, cls)
    if not os.path.exists(cls_path):
        continue
    for video_name in os.listdir(cls_path):
        video_path = os.path.join(cls_path, video_name)
        if len(os.listdir(video_path)) == 16:
            all_samples.append((video_path, class_to_idx[cls]))

# Split
random.shuffle(all_samples)
split         = int(0.8 * len(all_samples))
train_samples = all_samples[:split]
val_samples   = all_samples[split:]

train_dataset = TrimodalDataset(train_samples, selected_classes,
                                 transform, training=True)
val_dataset   = TrimodalDataset(val_samples,   selected_classes,
                                 transform, training=False)

# ✅ num_workers=0 — BERT + gTTS both need main process
train_loader = DataLoader(train_dataset, batch_size=8,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=8,
                          shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, trimodal_model.parameters()),
    lr=0.0001
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

history = {'train_loss': [], 'val_loss': [],
           'train_acc' : [], 'val_acc' : []}

best_val_acc    = 0.0
EPOCHS          = 15
checkpoint_path = '/content/drive/MyDrive/NewsImageCNN/checkpoints/trimodal_best.pth'

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 40)

    for phase in ['train', 'val']:
        if phase == 'train':
            trimodal_model.train()
            loader = train_loader
        else:
            trimodal_model.eval()
            loader = val_loader

        running_loss     = 0.0
        running_corrects = 0

        for videos, text_feats, audio_feats, labels in loader:
            videos      = videos.to(device)
            text_feats  = text_feats.to(device)
            audio_feats = audio_feats.to(device)
            labels      = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs  = trimodal_model(videos, text_feats, audio_feats)
                loss     = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss     += loss.item() * videos.size(0)
            running_corrects += torch.sum(preds == labels.data)

        if phase == 'train':
            scheduler.step()

        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc  = running_corrects.double() / len(loader.dataset)

        print(f"{phase.upper()} — Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}")

        if phase == 'train':
            history['train_loss'].append(epoch_loss)
            history['train_acc'].append(epoch_acc.item())
        else:
            history['val_loss'].append(epoch_loss)
            history['val_acc'].append(epoch_acc.item())

            if epoch_acc > best_val_acc:
                best_val_acc = epoch_acc
                torch.save(trimodal_model.state_dict(), checkpoint_path)
                print(f"  ✅ Best model saved! Val Acc: {best_val_acc:.4f}")

print(f"\nTraining Complete! Best Val Acc: {best_val_acc:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load best model
trimodal_model.load_state_dict(torch.load(checkpoint_path))
trimodal_model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for videos, text_feats, audio_feats, labels in val_loader:
        videos      = videos.to(device)
        text_feats  = text_feats.to(device)
        audio_feats = audio_feats.to(device)
        outputs     = trimodal_model(videos, text_feats, audio_feats)
        _, preds    = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Report
print("Classification Report:")
print(classification_report(all_labels, all_preds,
                             target_names=selected_classes))

# Plots
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

axes[0].plot(history['train_acc'], label='Train Acc', marker='o')
axes[0].plot(history['val_acc'],   label='Val Acc',   marker='o')
axes[0].set_title('Trimodal Fusion Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_loss'], label='Train Loss', marker='o')
axes[1].plot(history['val_loss'],   label='Val Loss',   marker='o')
axes[1].set_title('Trimodal Fusion Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=selected_classes,
            yticklabels=selected_classes, ax=axes[2])
axes[2].set_title('Confusion Matrix — Trimodal')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/NewsImageCNN/trimodal_results.png', dpi=150)
plt.show()